# LangChain Complete Guide: From Fundamentals to Advanced RAG

This notebook covers:
1. **Setup & Configuration** - Dependencies and OpenRouter/OpenAI setup
2. **LangChain Fundamentals** - LLM calls, Prompts, Output Parsers
3. **Memory Systems** - Buffer, Window, Summary, Entity Memory
4. **Chains** - LCEL, Sequential, Router chains
5. **Document Loading** - PDF, CSV, TXT, TSV support
6. **Vector Stores & Embeddings** - ChromaDB setup
7. **Basic RAG** - Simple retrieval-augmented generation
8. **Advanced RAG** - MMR, compression, conversation RAG
9. **Tool Calling & Agents** - Custom tools, RAG as a tool
10. **Complete Application** - Putting it all together
11. **HR RAG Chatbot with UI** - Interactive Gradio-based HR assistant

---

## Section 1: Setup & Configuration

In [1]:
# Install required dependencies
!pip install -q \
    langchain>=0.3.0 \
    langchain-openai>=0.2.0 \
    langchain-community>=0.3.0 \
    langchain-core>=0.3.0 \
    chromadb>=0.5.0 \
    pypdf>=4.0 \
    python-dotenv>=1.0 \
    pandas>=2.0 \
    tiktoken>=0.7.0 \
    sentence-transformers>=3.0 \
    gradio>=4.0 \
    fpdf2>=2.7.0

zsh:1: 0.3.0 not found


In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ============================================================
# CONFIGURATION: Choose ONE of the options below
# ============================================================

# OPTION 1: OpenRouter (supports multiple models - GPT, Claude, etc.)
# Set OPENROUTER_API_KEY in your .env file
USE_OPENROUTER = True  # Set to False for direct OpenAI

if USE_OPENROUTER:
    API_KEY = os.getenv("OPENROUTER_API_KEY", "your-openrouter-api-key")
    BASE_URL = "https://openrouter.ai/api/v1"
    MODEL_NAME = "openai/gpt-4o-mini"  # or "anthropic/claude-3.5-sonnet"
    print("Using OpenRouter API")
else:
    # OPTION 2: Direct OpenAI
    # Set OPENAI_API_KEY in your .env file
    API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")
    BASE_URL = None  # Uses default OpenAI endpoint
    MODEL_NAME = "gpt-4o-mini"  # or "gpt-4o", "gpt-3.5-turbo"
    print("Using OpenAI API directly")

print("Configuration loaded successfully!")

Using OpenRouter API
Configuration loaded successfully!


In [3]:
from langchain_openai import ChatOpenAI

# Initialize the LLM (works with both OpenRouter and OpenAI)
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=API_KEY,
    openai_api_base=BASE_URL,  # None for OpenAI, URL for OpenRouter
    temperature=0.7,
    max_tokens=1000,
)

# Test the connection
response = llm.invoke("Say 'Hello, LangChain!' in a creative way.")
print(response.content)

Greetings, vibrant tapestry of language and code! Hello, LangChain! 🌟


---
## Section 2: LangChain Fundamentals

### 2.1 LLM Calls

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Method 1: Simple string invocation
response = llm.invoke("What is the capital of France?")
print("Simple invocation:")
print(response.content)
print()

Simple invocation:
The capital of France is Paris.



In [5]:
# Method 2: Using message objects for more control
messages = [
    SystemMessage(content="You are a helpful geography expert. Be concise."),
    HumanMessage(content="What is the capital of France?")
]

response = llm.invoke(messages)
print("With system message:")
print(response.content)

With system message:
The capital of France is Paris.


In [6]:
# Method 3: Batch processing multiple inputs
questions = [
    "What is 2 + 2?",
    "What color is the sky?",
    "Who wrote Romeo and Juliet?"
]

responses = llm.batch(questions)
print("Batch processing:")
for q, r in zip(questions, responses):
    print(f"Q: {q}")
    print(f"A: {r.content}\n")

Batch processing:
Q: What is 2 + 2?
A: 2 + 2 equals 4.

Q: What color is the sky?
A: The color of the sky is typically blue during the day due to the scattering of sunlight by the Earth's atmosphere. This scattering is more effective at shorter wavelengths, which correspond to blue light. However, the sky can also appear in various colors at different times, such as red or orange during sunrise and sunset, or gray when it's overcast. At night, the sky appears black, dotted with stars.

Q: Who wrote Romeo and Juliet?
A: "Romeo and Juliet" was written by William Shakespeare. It is one of his most famous plays and was likely composed in the early 1590s.



In [7]:
# Method 4: Streaming responses
print("Streaming response:")
for chunk in llm.stream("Tell me a short joke about programming."):
    print(chunk.content, end="", flush=True)
print()

Streaming response:
Why do programmers prefer dark mode?

Because light attracts bugs!


### 2.2 Prompts & Prompt Templates

In [8]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate

# Simple PromptTemplate
simple_template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms for a beginner."
)

# Format the prompt
formatted = simple_template.format(topic="machine learning")
print("Formatted prompt:")
print(formatted)
print()

# Use with LLM
response = llm.invoke(formatted)
print("Response:")
print(response.content)

Formatted prompt:
Explain machine learning in simple terms for a beginner.

Response:
Sure! Machine learning is a way for computers to learn from data and make decisions or predictions without being explicitly programmed for every task. Here’s a simple breakdown:

1. **Learning from Data**: Just like humans learn from experience, machines learn from data. For example, if you show a computer lots of pictures of cats and dogs, it can learn to recognize the differences between them.

2. **Training**: To get started, we provide the machine with a large set of examples (this is called the training data). The computer analyzes this data to find patterns. For instance, it might notice that cats often have pointy ears while dogs usually have floppy ears.

3. **Making Predictions**: After the machine has learned from the training data, we can give it new examples it hasn’t seen before, and it can make predictions about them. So, if we show it a new picture, it can tell us whether it thinks it’s

In [9]:
# ChatPromptTemplate for chat models (recommended)
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Always respond in a {tone} manner."),
    ("human", "{question}")
])

# Format and invoke
messages = chat_template.format_messages(
    role="helpful coding assistant",
    tone="friendly and encouraging",
    question="How do I write a for loop in Python?"
)

response = llm.invoke(messages)
print(response.content)

Writing a for loop in Python is quite simple! A for loop allows you to iterate over a sequence (like a list, tuple, string, or range) and execute a block of code for each item in that sequence. Here’s the basic structure:

```python
for item in iterable:
    # Do something with item
    print(item)
```

### Example 1: Looping through a list
Here’s an example where we loop through a list of fruits:

```python
fruits = ['apple', 'banana', 'cherry']

for fruit in fruits:
    print(fruit)
```

In this example, `fruit` takes the value of each item in the `fruits` list one by one, and the loop prints each fruit.

### Example 2: Using `range()`
You can also use a for loop with the `range()` function to repeat an action a specific number of times:

```python
for i in range(5):
    print(i)
```

This will print numbers from 0 to 4.

### Example 3: Looping through a string
You can iterate through each character in a string too:

```python
word = "hello"

for letter in word:
    print(letter)
```

In [10]:
# FewShotPromptTemplate - teaching the model with examples
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "fast", "output": "slow"},
]

example_template = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Give the opposite of each word.",
    suffix="Input: {word}\nOutput:",
    input_variables=["word"],
)

formatted = few_shot_prompt.format(word="bright")
print("Few-shot prompt:")
print(formatted)
print()

response = llm.invoke(formatted)
print("Model's answer:", response.content)

Few-shot prompt:
Give the opposite of each word.

Input: happy
Output: sad

Input: tall
Output: short

Input: fast
Output: slow

Input: bright
Output:

Model's answer: dull


### 2.3 Output Parsers

In [11]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, CommaSeparatedListOutputParser
from pydantic import BaseModel, Field
from typing import List

# 1. StrOutputParser - extracts string content from response
str_parser = StrOutputParser()

# Chain: prompt -> llm -> parser
chain = chat_template | llm | str_parser

result = chain.invoke({
    "role": "Python expert",
    "tone": "concise",
    "question": "What is a list comprehension?"
})

print("StrOutputParser result:")
print(result)  # Now it's a string, not an AIMessage object
print(f"Type: {type(result)}")

StrOutputParser result:
A list comprehension is a concise way to create lists in Python using a single line of code. It consists of brackets containing an expression followed by a `for` clause, and can include optional `if` clauses to filter items.

Example:

```python
squares = [x**2 for x in range(10) if x % 2 == 0]
```

This creates a list of squares of even numbers from 0 to 9.
Type: <class 'str'>


In [12]:
# 2. CommaSeparatedListOutputParser
list_parser = CommaSeparatedListOutputParser()

list_prompt = PromptTemplate(
    template="List 5 {category}. Respond with ONLY a comma-separated list, nothing else.",
    input_variables=["category"]
)

chain = list_prompt | llm | list_parser
result = chain.invoke({"category": "programming languages"})

print("CommaSeparatedListOutputParser result:")
print(result)
print(f"Type: {type(result)}")

CommaSeparatedListOutputParser result:
['Python', 'Java', 'C++', 'JavaScript', 'Ruby']
Type: <class 'list'>


In [13]:
# 3. JsonOutputParser with Pydantic model
class Book(BaseModel):
    """Information about a book."""
    title: str = Field(description="The title of the book")
    author: str = Field(description="The author of the book")
    year: int = Field(description="The year the book was published")
    genres: List[str] = Field(description="List of genres")

json_parser = JsonOutputParser(pydantic_object=Book)

json_prompt = PromptTemplate(
    template="""Generate information about a famous {genre} book.
    
{format_instructions}

Respond with ONLY the JSON, no additional text.""",
    input_variables=["genre"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

chain = json_prompt | llm | json_parser
result = chain.invoke({"genre": "science fiction"})

print("JsonOutputParser result:")
print(result)
print(f"Type: {type(result)}")
print(f"Title: {result['title']}")

JsonOutputParser result:
{'title': 'Dune', 'author': 'Frank Herbert', 'year': 1965, 'genres': ['Science Fiction', 'Adventure', 'Fantasy']}
Type: <class 'dict'>
Title: Dune


In [14]:
# 4. PydanticOutputParser - returns actual Pydantic objects
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class MovieReview(BaseModel):
    """A movie review."""
    movie_title: str = Field(description="Title of the movie")
    rating: int = Field(description="Rating from 1-10")
    summary: str = Field(description="Brief summary of the review")
    recommended: bool = Field(description="Whether you'd recommend this movie")

pydantic_parser = PydanticOutputParser(pydantic_object=MovieReview)

review_prompt = PromptTemplate(
    template="""Write a review for the movie "{movie}".
    
{format_instructions}""",
    input_variables=["movie"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

chain = review_prompt | llm | pydantic_parser
result = chain.invoke({"movie": "The Matrix"})

print("PydanticOutputParser result:")
print(f"Movie: {result.movie_title}")
print(f"Rating: {result.rating}/10")
print(f"Summary: {result.summary}")
print(f"Recommended: {result.recommended}")

PydanticOutputParser result:
Movie: The Matrix
Rating: 10/10
Summary: A groundbreaking sci-fi film that combines philosophy, action, and stunning visuals. The Matrix challenges perceptions of reality and has left a lasting impact on the genre.
Recommended: True


---
## Section 3: Memory Systems

Memory allows LLMs to remember previous interactions within a conversation.

In [15]:
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    ConversationSummaryBufferMemory,
    ConversationEntityMemory
)
from langchain_classic.chains import ConversationChain

ModuleNotFoundError: No module named 'langchain.memory'

### 3.1 ConversationBufferMemory
Stores the entire conversation history. Simple but can grow unbounded.

In [ ]:
# ConversationBufferMemory - stores full conversation history
buffer_memory = ConversationBufferMemory(return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=buffer_memory,
    verbose=True  # Shows what's happening internally
)

# Have a conversation
print(conversation.predict(input="Hi, my name is Alice."))
print("---")
print(conversation.predict(input="What's a good restaurant in Paris?"))
print("---")
print(conversation.predict(input="What's my name?"))  # Should remember!

In [ ]:
# Inspect the memory
print("Memory contents:")
print(buffer_memory.load_memory_variables({}))

### 3.2 ConversationBufferWindowMemory
Keeps only the last K exchanges. Good for long conversations where only recent context matters.

In [ ]:
# ConversationBufferWindowMemory - keeps last k messages
window_memory = ConversationBufferWindowMemory(k=2, return_messages=True)

conversation_window = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=False
)

# Simulate a longer conversation
print("Message 1:", conversation_window.predict(input="My favorite color is blue."))
print("\nMessage 2:", conversation_window.predict(input="I also love pizza."))
print("\nMessage 3:", conversation_window.predict(input="My dog's name is Max."))
print("\nMessage 4:", conversation_window.predict(input="What was my favorite color?"))  # Might not remember!

print("\nMemory (only last 2 exchanges):")
print(window_memory.load_memory_variables({}))

### 3.3 ConversationSummaryMemory
Uses an LLM to create a running summary of the conversation. Great for very long conversations.

In [ ]:
# ConversationSummaryMemory - summarizes conversation
summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)

conversation_summary = ConversationChain(
    llm=llm,
    memory=summary_memory,
    verbose=False
)

# Have a multi-turn conversation
conversation_summary.predict(input="I'm planning a trip to Japan next month.")
conversation_summary.predict(input="I want to visit Tokyo and Kyoto.")
conversation_summary.predict(input="I'm interested in temples and good food.")

print("Summary of conversation:")
print(summary_memory.load_memory_variables({}))

### 3.4 ConversationSummaryBufferMemory
Hybrid approach: keeps recent messages AND maintains a summary of older ones. Best of both worlds.

In [ ]:
# ConversationSummaryBufferMemory - hybrid approach
summary_buffer_memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=100,  # Summarize when exceeding this limit
    return_messages=True
)

conversation_hybrid = ConversationChain(
    llm=llm,
    memory=summary_buffer_memory,
    verbose=False
)

# Long conversation
conversation_hybrid.predict(input="I'm a software engineer working on AI projects.")
conversation_hybrid.predict(input="Currently I'm building a RAG system.")
conversation_hybrid.predict(input="I use Python and LangChain.")
conversation_hybrid.predict(input="What frameworks should I also consider?")

print("Memory state (summary + recent buffer):")
print(summary_buffer_memory.load_memory_variables({}))

### 3.5 Entity Memory
Extracts and tracks entities (people, places, concepts) mentioned in the conversation.

In [ ]:
# ConversationEntityMemory - tracks entities
entity_memory = ConversationEntityMemory(llm=llm)

conversation_entity = ConversationChain(
    llm=llm,
    memory=entity_memory,
    verbose=False
)

# Conversation with entities
conversation_entity.predict(input="John works at Google as a machine learning engineer.")
conversation_entity.predict(input="His colleague Sarah specializes in NLP.")
conversation_entity.predict(input="They're building a chatbot together.")

print("Tracked entities:")
print(entity_memory.entity_store.store)

---
## Section 4: Chains

Chains allow you to combine multiple components (prompts, LLMs, parsers, tools) into a single pipeline.

### 4.1 LCEL (LangChain Expression Language)
Modern, recommended way to build chains using the pipe (`|`) operator.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# Simple LCEL chain: prompt | llm | parser
simple_chain = (
    ChatPromptTemplate.from_template("Tell me a fun fact about {topic}")
    | llm
    | StrOutputParser()
)

result = simple_chain.invoke({"topic": "octopuses"})
print("Simple chain result:")
print(result)

In [ ]:
# RunnablePassthrough - passes input through unchanged
# Useful when you need original input alongside processed data

chain_with_passthrough = RunnableParallel(
    original_topic=RunnablePassthrough(),
    fun_fact=ChatPromptTemplate.from_template("Tell me one fun fact about {topic}") | llm | StrOutputParser()
)

result = chain_with_passthrough.invoke({"topic": "Python"})
print("With passthrough:")
print(f"Original: {result['original_topic']}")
print(f"Fun fact: {result['fun_fact']}")

In [ ]:
# RunnableParallel - run multiple chains in parallel
parallel_chain = RunnableParallel(
    summary=ChatPromptTemplate.from_template("Summarize {topic} in one sentence") | llm | StrOutputParser(),
    key_points=ChatPromptTemplate.from_template("List 3 key points about {topic}") | llm | StrOutputParser(),
    use_cases=ChatPromptTemplate.from_template("Give 2 practical use cases for {topic}") | llm | StrOutputParser()
)

result = parallel_chain.invoke({"topic": "Docker containers"})
print("Parallel execution results:")
for key, value in result.items():
    print(f"\n{key.upper()}:")
    print(value)

In [ ]:
# RunnableLambda - use custom functions in chains
def format_output(data: dict) -> str:
    """Custom function to format the output."""
    return f"""
=== TOPIC ANALYSIS ===

SUMMARY:
{data['summary']}

KEY POINTS:
{data['key_points']}

USE CASES:
{data['use_cases']}
=====================
"""

full_chain = parallel_chain | RunnableLambda(format_output)

result = full_chain.invoke({"topic": "REST APIs"})
print(result)

### 4.2 Sequential Chains
Output of one step becomes input to the next.

In [ ]:
# Sequential chain: research -> outline -> write

# Step 1: Generate research notes
research_prompt = ChatPromptTemplate.from_template(
    "Research the topic '{topic}' and provide 3 key insights. Be concise."
)

# Step 2: Create outline based on research
outline_prompt = ChatPromptTemplate.from_template(
    """Based on these research notes:
{research}

Create a brief outline for a blog post."""
)

# Step 3: Write introduction based on outline
intro_prompt = ChatPromptTemplate.from_template(
    """Based on this outline:
{outline}

Write a compelling introduction paragraph (2-3 sentences)."""
)

# Build the sequential chain
sequential_chain = (
    {"topic": RunnablePassthrough()}
    | RunnableParallel(
        topic=lambda x: x["topic"],
        research=research_prompt | llm | StrOutputParser()
    )
    | RunnableParallel(
        research=lambda x: x["research"],
        outline=outline_prompt | llm | StrOutputParser()
    )
    | RunnableParallel(
        outline=lambda x: x["outline"],
        introduction=intro_prompt | llm | StrOutputParser()
    )
)

result = sequential_chain.invoke({"topic": "sustainable energy"})
print("OUTLINE:")
print(result["outline"])
print("\nINTRODUCTION:")
print(result["introduction"])

### 4.3 Router Chain
Routes inputs to different chains based on conditions.

In [ ]:
from langchain_core.runnables import RunnableBranch

# Different prompts for different types of questions
math_prompt = ChatPromptTemplate.from_template(
    "You are a math tutor. Solve this problem step by step: {question}"
)

science_prompt = ChatPromptTemplate.from_template(
    "You are a science expert. Explain this concept clearly: {question}"
)

history_prompt = ChatPromptTemplate.from_template(
    "You are a history professor. Provide context and details about: {question}"
)

general_prompt = ChatPromptTemplate.from_template(
    "Answer this question helpfully: {question}"
)

# Classification chain to determine question type
classify_prompt = ChatPromptTemplate.from_template(
    """Classify this question into one category: math, science, history, or general.
Question: {question}
Respond with ONLY the category name, nothing else."""
)

def route_question(info):
    """Route to appropriate chain based on classification."""
    category = info["category"].lower().strip()
    question = info["question"]
    
    if "math" in category:
        return math_prompt.format(question=question)
    elif "science" in category:
        return science_prompt.format(question=question)
    elif "history" in category:
        return history_prompt.format(question=question)
    else:
        return general_prompt.format(question=question)

# Full routing chain
router_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        category=classify_prompt | llm | StrOutputParser()
    )
    | RunnableLambda(route_question)
    | llm
    | StrOutputParser()
)

# Test with different types of questions
questions = [
    "What is 15% of 240?",
    "How does photosynthesis work?",
    "Who was Cleopatra?"
]

for q in questions:
    print(f"Q: {q}")
    result = router_chain.invoke(q)
    print(f"A: {result}\n")

---
## Section 5: Document Loading & Processing

### 5.1 Document Loaders

In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    TextLoader,
    DirectoryLoader,
    UnstructuredFileLoader
)
from langchain.schema import Document
import os

# Create sample data directory
os.makedirs("sample_data", exist_ok=True)

In [ ]:
# Create sample files for demonstration

# Sample TXT file
txt_content = """Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience.
There are three main types of machine learning:
1. Supervised Learning - Learning from labeled data
2. Unsupervised Learning - Finding patterns in unlabeled data  
3. Reinforcement Learning - Learning through trial and error

Common applications include image recognition, natural language processing, and recommendation systems.
"""

with open("sample_data/ml_intro.txt", "w") as f:
    f.write(txt_content)

# Sample CSV file
csv_content = """name,role,department,years_experience
Alice,Engineer,Engineering,5
Bob,Designer,Design,3
Carol,Manager,Engineering,8
David,Analyst,Data Science,4
"""

with open("sample_data/employees.csv", "w") as f:
    f.write(csv_content)

# Sample TSV file
tsv_content = """product\tprice\tcategory\tin_stock
Laptop\t999.99\tElectronics\tTrue
Headphones\t149.99\tElectronics\tTrue
Desk\t299.99\tFurniture\tFalse
Monitor\t399.99\tElectronics\tTrue
"""

with open("sample_data/products.tsv", "w") as f:
    f.write(tsv_content)

print("Sample files created in sample_data/")

In [ ]:
# TextLoader - for .txt files
txt_loader = TextLoader("sample_data/ml_intro.txt")
txt_docs = txt_loader.load()

print("TXT Document:")
print(f"Content: {txt_docs[0].page_content[:200]}...")
print(f"Metadata: {txt_docs[0].metadata}")

In [ ]:
# CSVLoader - for .csv files (each row becomes a document)
csv_loader = CSVLoader("sample_data/employees.csv")
csv_docs = csv_loader.load()

print("CSV Documents:")
for i, doc in enumerate(csv_docs[:2]):
    print(f"\nDocument {i+1}:")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

In [ ]:
# TSV Loader - custom approach using pandas
import pandas as pd

def load_tsv(file_path: str) -> list[Document]:
    """Load TSV file as documents."""
    df = pd.read_csv(file_path, sep='\t')
    documents = []
    for idx, row in df.iterrows():
        content = " | ".join([f"{col}: {val}" for col, val in row.items()])
        doc = Document(
            page_content=content,
            metadata={"source": file_path, "row": idx}
        )
        documents.append(doc)
    return documents

tsv_docs = load_tsv("sample_data/products.tsv")
print("TSV Documents:")
for doc in tsv_docs:
    print(f"Content: {doc.page_content}")

In [ ]:
# Universal loader function for multiple file types
def load_document(file_path: str) -> list[Document]:
    """
    Load a document based on its file extension.
    Supports: PDF, CSV, TXT, TSV
    """
    extension = file_path.lower().split('.')[-1]
    
    if extension == 'pdf':
        loader = PyPDFLoader(file_path)
    elif extension == 'csv':
        loader = CSVLoader(file_path)
    elif extension == 'txt':
        loader = TextLoader(file_path)
    elif extension == 'tsv':
        return load_tsv(file_path)
    else:
        # Fallback to unstructured loader
        loader = UnstructuredFileLoader(file_path)
    
    return loader.load()

# Test with different file types
print("Universal loader test:")
for file in ["sample_data/ml_intro.txt", "sample_data/employees.csv", "sample_data/products.tsv"]:
    docs = load_document(file)
    print(f"\n{file}: Loaded {len(docs)} document(s)")

### 5.2 Text Splitting
Large documents need to be split into smaller chunks for effective retrieval.

In [ ]:
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

# Sample long text for demonstration
long_text = """
Artificial Intelligence: A Comprehensive Overview

Chapter 1: Introduction
Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, 
especially computer systems. These processes include learning, reasoning, and self-correction.

Chapter 2: History
The term "artificial intelligence" was coined in 1956 at a conference at Dartmouth College. 
Since then, AI has evolved through several phases, from symbolic AI to machine learning 
and now to deep learning.

Chapter 3: Applications
AI is used in various fields including healthcare for diagnosis, finance for fraud detection,
transportation for autonomous vehicles, and entertainment for recommendation systems.

Chapter 4: Future
The future of AI holds promise for more advanced applications including artificial general
intelligence (AGI), which would match human cognitive abilities across all domains.
"""

# RecursiveCharacterTextSplitter (Recommended)
# Tries to split on paragraphs, then sentences, then words
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = recursive_splitter.split_text(long_text)
print(f"RecursiveCharacterTextSplitter: {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(chunk[:100] + "...")

In [ ]:
# CharacterTextSplitter - splits only on a specific separator
char_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separator="\n"
)

char_chunks = char_splitter.split_text(long_text)
print(f"CharacterTextSplitter: {len(char_chunks)} chunks")

In [ ]:
# Split documents (not just text)
from langchain.schema import Document

doc = Document(page_content=long_text, metadata={"source": "ai_overview.txt"})

split_docs = recursive_splitter.split_documents([doc])
print(f"Split into {len(split_docs)} document chunks")
print(f"\nFirst chunk metadata: {split_docs[0].metadata}")

---
## Section 6: Vector Stores & Embeddings

### 6.1 Embeddings

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Using HuggingFace embeddings (free, runs locally)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Test embedding
text = "Machine learning is a subset of artificial intelligence."
embedding = embeddings.embed_query(text)

print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

In [ ]:
# Compare similarity of embeddings
import numpy as np

texts = [
    "Machine learning is a subset of artificial intelligence.",
    "AI and ML are related fields in computer science.",
    "I love eating pizza on Fridays."
]

embeds = [embeddings.embed_query(t) for t in texts]

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine similarities:")
print(f"Text 1 vs Text 2 (related): {cosine_similarity(embeds[0], embeds[1]):.4f}")
print(f"Text 1 vs Text 3 (unrelated): {cosine_similarity(embeds[0], embeds[2]):.4f}")

### 6.2 Vector Stores with ChromaDB

In [ ]:
from langchain_community.vectorstores import Chroma

# Sample documents for the vector store
documents = [
    Document(page_content="Python is a versatile programming language used for web development, data science, and AI.", 
             metadata={"topic": "programming", "language": "Python"}),
    Document(page_content="JavaScript is essential for web development and runs in browsers.", 
             metadata={"topic": "programming", "language": "JavaScript"}),
    Document(page_content="Machine learning models learn patterns from data to make predictions.", 
             metadata={"topic": "AI", "subtopic": "ML"}),
    Document(page_content="Deep learning uses neural networks with many layers for complex tasks.", 
             metadata={"topic": "AI", "subtopic": "DL"}),
    Document(page_content="Natural Language Processing enables computers to understand human language.", 
             metadata={"topic": "AI", "subtopic": "NLP"}),
    Document(page_content="Docker containers package applications with their dependencies for consistent deployment.", 
             metadata={"topic": "DevOps", "tool": "Docker"}),
    Document(page_content="Kubernetes orchestrates container deployment, scaling, and management.", 
             metadata={"topic": "DevOps", "tool": "Kubernetes"}),
]

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Persist to disk
)

print(f"Created vector store with {vectorstore._collection.count()} documents")

In [ ]:
# Similarity search
query = "How do I build AI applications?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
print("Top 3 similar documents:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. {doc.page_content}")
    print(f"   Metadata: {doc.metadata}")

In [ ]:
# Similarity search with scores
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print("Results with similarity scores (lower = more similar):")
for doc, score in results_with_scores:
    print(f"Score: {score:.4f} - {doc.page_content[:50]}...")

In [ ]:
# Filtered search using metadata
results = vectorstore.similarity_search(
    "What tools are used?",
    k=3,
    filter={"topic": "DevOps"}
)

print("Filtered search (DevOps only):")
for doc in results:
    print(f"- {doc.page_content}")

In [ ]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Test retriever
docs = retriever.invoke("Tell me about programming languages")
print("Retriever results:")
for doc in docs:
    print(f"- {doc.page_content}")

---
## Section 7: Basic RAG Implementation

### 7.1 Simple RAG Pipeline

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# RAG prompt template
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context. If the context doesn't contain 
enough information, say "I don't have enough information to answer this question."

Context:
{context}

Question: {question}

Answer:
""")

# Helper function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test the RAG chain
question = "What programming language is good for data science?"
answer = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"\nAnswer: {answer}")

In [ ]:
# Test with more questions
questions = [
    "What is deep learning used for?",
    "How does Docker help with deployment?",
    "What's the difference between ML and DL?",
    "What is the best pizza topping?"  # Should admit it doesn't know
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}\n")

### 7.2 RAG with Multiple File Types

In [ ]:
# Load all sample documents
all_docs = []

# Load different file types
files_to_load = [
    "sample_data/ml_intro.txt",
    "sample_data/employees.csv",
    "sample_data/products.tsv"
]

for file_path in files_to_load:
    docs = load_document(file_path)
    all_docs.extend(docs)
    print(f"Loaded {len(docs)} document(s) from {file_path}")

print(f"\nTotal documents: {len(all_docs)}")

In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(all_docs)
print(f"Split into {len(split_docs)} chunks")

In [ ]:
# Create new vector store with all documents
multi_vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_multi_db"
)

multi_retriever = multi_vectorstore.as_retriever(search_kwargs={"k": 4})

# New RAG chain with the multi-file retriever
multi_rag_chain = (
    {
        "context": multi_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Test multi-file RAG
test_questions = [
    "What are the types of machine learning?",
    "Who works in the Engineering department?",
    "What electronics products are available?",
    "How much does the laptop cost?"
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {multi_rag_chain.invoke(q)}\n")

---
## Section 8: Advanced RAG Techniques

### 8.1 Retrieval Strategies

In [ ]:
# MMR (Maximum Marginal Relevance) - balances relevance with diversity
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 10,  # Fetch more, then select diverse subset
        "lambda_mult": 0.5  # 0 = max diversity, 1 = max relevance
    }
)

print("MMR Retrieval Results:")
mmr_docs = mmr_retriever.invoke("Tell me about AI and programming")
for doc in mmr_docs:
    print(f"- {doc.page_content[:80]}...")

In [ ]:
# Similarity score threshold retriever
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.5,  # Only return docs above this similarity
        "k": 5
    }
)

print("Threshold-based retrieval:")
threshold_docs = threshold_retriever.invoke("What is Python?")
print(f"Found {len(threshold_docs)} documents above threshold")

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Contextual Compression - extracts only relevant parts of documents
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

print("Compressed retrieval (extracts relevant parts):")
compressed_docs = compression_retriever.invoke("What is used for AI?")
for doc in compressed_docs:
    print(f"- {doc.page_content}")

In [ ]:
from langchain.retrievers import MultiQueryRetriever

# Multi-Query Retriever - generates multiple queries for better recall
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

print("Multi-query retrieval:")
# This generates multiple variations of the query internally
multi_docs = multi_query_retriever.invoke("How do I build smart applications?")
print(f"Retrieved {len(multi_docs)} unique documents")
for doc in multi_docs:
    print(f"- {doc.page_content[:60]}...")

### 8.2 RAG Chain Patterns

In [ ]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Stuff Documents Chain - puts all documents into context (default pattern)
stuff_prompt = ChatPromptTemplate.from_template("""
Answer the question based on the following context:

{context}

Question: {input}
""")

stuff_chain = create_stuff_documents_chain(llm, stuff_prompt)
retrieval_chain = create_retrieval_chain(retriever, stuff_chain)

result = retrieval_chain.invoke({"input": "What is machine learning?"})
print("Stuff Documents Chain:")
print(f"Answer: {result['answer']}")
print(f"\nSource documents used: {len(result['context'])}")

In [ ]:
from langchain.chains.summarize import load_summarize_chain

# Map-Reduce Chain - processes documents individually, then combines
# Good for: Large number of documents, summarization tasks

map_reduce_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce",
    verbose=False
)

# Get some documents
docs_to_summarize = retriever.invoke("Tell me about technology")

if docs_to_summarize:
    summary = map_reduce_chain.invoke(docs_to_summarize)
    print("Map-Reduce Summary:")
    print(summary['output_text'])

In [ ]:
# Refine Chain - iteratively refines answer with each document
# Good for: High-quality answers, when context from multiple docs is needed

refine_chain = load_summarize_chain(
    llm,
    chain_type="refine",
    verbose=False
)

if docs_to_summarize:
    refined_summary = refine_chain.invoke(docs_to_summarize)
    print("Refine Chain Summary:")
    print(refined_summary['output_text'])

### 8.3 Conversation RAG (with Chat History)

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# Contextualize question based on chat history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", """Given the chat history and a follow-up question, 
    rephrase the question to be standalone (understandable without the chat history).
    If the question is already standalone, return it as-is."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Create history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

In [ ]:
# QA prompt that includes chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer questions based on the context provided.
    If you don't know, say so. Be concise.
    
    Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Build the conversational RAG chain
qa_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag = create_retrieval_chain(history_aware_retriever, qa_chain)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Simulate a conversation
chat_history = []

# First question
q1 = "What programming languages are mentioned?"
result1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {result1['answer']}\n")

# Update chat history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=result1['answer'])
])

# Follow-up question (uses "it" which requires context)
q2 = "What can you do with it?"  # Refers to Python from previous answer
result2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})

print(f"Q2: {q2}")
print(f"A2: {result2['answer']}")

---
## Section 9: Tool Calling & Agents

### 9.1 Defining Tools

In [ ]:
from langchain.tools import tool, StructuredTool
from pydantic import BaseModel, Field
from typing import Optional
import math

# Simple tool using @tool decorator
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Use this for any math calculations.
    
    Args:
        expression: A mathematical expression to evaluate (e.g., '2 + 2', '10 * 5')
    """
    try:
        # Safe evaluation of mathematical expressions
        allowed_names = {"sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, "pi": math.pi}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_current_time() -> str:
    """Get the current date and time. No arguments needed."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Test the tools
print("Calculator tool:")
print(calculate.invoke("15 * 7 + 3"))

print("\nTime tool:")
print(get_current_time.invoke(""))

In [ ]:
# StructuredTool for complex inputs
class WeatherInput(BaseModel):
    """Input for the weather tool."""
    city: str = Field(description="The city to get weather for")
    unit: str = Field(default="celsius", description="Temperature unit: 'celsius' or 'fahrenheit'")

def get_weather(city: str, unit: str = "celsius") -> str:
    """Simulated weather lookup."""
    # In real app, this would call a weather API
    weather_data = {
        "new york": {"temp_c": 22, "condition": "Sunny"},
        "london": {"temp_c": 15, "condition": "Cloudy"},
        "tokyo": {"temp_c": 28, "condition": "Humid"},
    }
    
    city_lower = city.lower()
    if city_lower in weather_data:
        data = weather_data[city_lower]
        temp = data["temp_c"]
        if unit == "fahrenheit":
            temp = temp * 9/5 + 32
            unit_symbol = "F"
        else:
            unit_symbol = "C"
        return f"Weather in {city}: {temp}°{unit_symbol}, {data['condition']}"
    else:
        return f"Weather data not available for {city}"

weather_tool = StructuredTool.from_function(
    func=get_weather,
    name="weather",
    description="Get the current weather for a city",
    args_schema=WeatherInput
)

print("Weather tool test:")
print(weather_tool.invoke({"city": "Tokyo", "unit": "celsius"}))

### 9.2 Creating RAG as a Tool

In [ ]:
# Wrap RAG pipeline as a tool
@tool
def search_knowledge_base(query: str) -> str:
    """Search the internal knowledge base for information about programming, 
    AI, DevOps, employees, and products. Use this when you need factual 
    information from the company database.
    
    Args:
        query: The question or search query about the knowledge base
    """
    # Use our multi-file RAG chain
    return multi_rag_chain.invoke(query)

# Test the RAG tool
print("RAG Tool test:")
print(search_knowledge_base.invoke("Who works in Engineering?"))

### 9.3 Building Agents

In [ ]:
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define our tools
tools = [calculate, get_current_time, weather_tool, search_knowledge_base]

# Agent prompt
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful AI assistant with access to various tools.
    
Available tools:
- calculate: For mathematical calculations
- get_current_time: To get the current date and time
- weather: To get weather information for a city
- search_knowledge_base: To search internal company data about employees, products, and tech topics

Use tools when needed. Think step by step. If multiple tools are needed, use them sequentially."""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Create the agent
agent = create_openai_tools_agent(llm, tools, agent_prompt)

# Create the executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Show reasoning steps
    handle_parsing_errors=True
)

In [ ]:
# Test the agent with different queries
print("Test 1: Math calculation")
result = agent_executor.invoke({"input": "What is 25% of 180?"})
print(f"Answer: {result['output']}\n")

In [ ]:
print("Test 2: Knowledge base search")
result = agent_executor.invoke({"input": "How much experience does Carol have?"})
print(f"Answer: {result['output']}\n")

In [ ]:
print("Test 3: Weather query")
result = agent_executor.invoke({"input": "What's the weather like in London?"})
print(f"Answer: {result['output']}\n")

In [ ]:
print("Test 4: Multi-step reasoning")
result = agent_executor.invoke({
    "input": "What is the laptop price in the product catalog, and calculate 10% discount on it."
})
print(f"Answer: {result['output']}")

### 9.4 Agent with Memory

In [ ]:
from langchain.memory import ConversationBufferMemory

# Create memory for the agent
agent_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Agent with memory
agent_with_memory = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=agent_memory,
    verbose=True,
    handle_parsing_errors=True
)

In [ ]:
# Conversation with memory
print("=== Conversation with Memory ===")

response1 = agent_with_memory.invoke({"input": "My name is Alice and I'm interested in machine learning."})
print(f"Response 1: {response1['output']}\n")

response2 = agent_with_memory.invoke({"input": "What types of machine learning are there?"})
print(f"Response 2: {response2['output']}\n")

response3 = agent_with_memory.invoke({"input": "What's my name again?"})
print(f"Response 3: {response3['output']}")

---
## Section 10: Complete Application

### 10.1 Building a Full RAG Agent System

In [ ]:
class RAGAgent:
    """
    A complete RAG Agent that combines:
    - Multi-file document loading
    - Vector store for retrieval
    - Multiple tools (calculator, weather, time)
    - Conversation memory
    """
    
    def __init__(self, llm, embeddings):
        self.llm = llm
        self.embeddings = embeddings
        self.vectorstore = None
        self.agent_executor = None
        self.memory = ConversationBufferWindowMemory(
            k=10,
            memory_key="chat_history",
            return_messages=True
        )
        
    def load_documents(self, file_paths: list[str]):
        """Load and index documents from multiple file types."""
        all_docs = []
        
        for file_path in file_paths:
            if os.path.exists(file_path):
                docs = load_document(file_path)
                all_docs.extend(docs)
                print(f"Loaded {len(docs)} doc(s) from {file_path}")
            else:
                print(f"Warning: {file_path} not found")
        
        # Split documents
        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        split_docs = splitter.split_documents(all_docs)
        
        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=split_docs,
            embedding=self.embeddings,
            persist_directory="./rag_agent_db"
        )
        
        print(f"Indexed {len(split_docs)} document chunks")
        return self
    
    def _create_rag_tool(self):
        """Create the RAG search tool."""
        retriever = self.vectorstore.as_retriever(search_kwargs={"k": 4})
        
        rag_prompt = ChatPromptTemplate.from_template("""
        Answer based on the context. If unsure, say so.
        
        Context: {context}
        Question: {question}
        """)
        
        rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | rag_prompt
            | self.llm
            | StrOutputParser()
        )
        
        @tool
        def search_documents(query: str) -> str:
            """Search the loaded documents for information. Use this for questions about 
            the content in uploaded files, company data, products, or employees."""
            return rag_chain.invoke(query)
        
        return search_documents
    
    def build_agent(self):
        """Build the agent with all tools."""
        if self.vectorstore is None:
            raise ValueError("Please load documents first using load_documents()")
        
        # Define all tools
        tools = [
            calculate,
            get_current_time,
            weather_tool,
            self._create_rag_tool()
        ]
        
        # Agent prompt
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a helpful AI assistant with access to tools and a document knowledge base.
            
Tools available:
- search_documents: Search uploaded documents for information
- calculate: Perform mathematical calculations
- get_current_time: Get current date/time
- weather: Get weather for a city

Be helpful, concise, and use tools when appropriate."""),
            MessagesPlaceholder(variable_name="chat_history", optional=True),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])
        
        agent = create_openai_tools_agent(self.llm, tools, prompt)
        
        self.agent_executor = AgentExecutor(
            agent=agent,
            tools=tools,
            memory=self.memory,
            verbose=False,
            handle_parsing_errors=True
        )
        
        print("Agent built successfully!")
        return self
    
    def chat(self, message: str) -> str:
        """Send a message to the agent."""
        if self.agent_executor is None:
            raise ValueError("Please build the agent first using build_agent()")
        
        result = self.agent_executor.invoke({"input": message})
        return result['output']
    
    def clear_memory(self):
        """Clear conversation memory."""
        self.memory.clear()
        print("Memory cleared.")

In [ ]:
# Create and configure the RAG Agent
rag_agent = RAGAgent(llm, embeddings)

# Load documents
rag_agent.load_documents([
    "sample_data/ml_intro.txt",
    "sample_data/employees.csv",
    "sample_data/products.tsv"
])

# Build the agent
rag_agent.build_agent()

In [ ]:
# Interactive chat loop
print("=== RAG Agent Chat ===")
print("Type 'quit' to exit, 'clear' to reset memory\n")

test_questions = [
    "What types of machine learning exist?",
    "Who are the engineers in the company?",
    "What's the most expensive product and calculate 20% off?",
    "What did we discuss about machine learning?"
]

for question in test_questions:
    print(f"You: {question}")
    response = rag_agent.chat(question)
    print(f"Agent: {response}\n")

### 10.2 Best Practices & Tips

In [ ]:
# Best Practices Summary

best_practices = """
=== LangChain Best Practices ===

1. PROMPT ENGINEERING
   - Be specific in system prompts
   - Use few-shot examples for complex tasks
   - Include output format instructions

2. MEMORY MANAGEMENT
   - Use BufferWindowMemory for long conversations
   - Use SummaryBufferMemory for very long sessions
   - Clear memory periodically to avoid context pollution

3. RAG OPTIMIZATION
   - Chunk size: 500-1000 chars for most use cases
   - Overlap: 10-20% of chunk size
   - Use MMR for diverse results
   - Consider hybrid search (keyword + semantic)

4. TOOL DESIGN
   - Write clear, specific tool descriptions
   - Include parameter descriptions
   - Handle errors gracefully
   - Return structured, parseable output

5. ERROR HANDLING
   - Wrap API calls in try/except
   - Set reasonable timeouts
   - Implement retry logic for transient failures
   - Log errors for debugging

6. COST OPTIMIZATION
   - Use smaller models for simple tasks
   - Cache embeddings when possible
   - Limit context size appropriately
   - Use streaming for long responses

7. TESTING
   - Test each component in isolation
   - Create test cases for edge cases
   - Monitor token usage
   - Track response quality metrics
"""

print(best_practices)

---
## Section 11: HR RAG Chatbot with Interactive UI

This section demonstrates a complete, production-ready HR chatbot with:
- Sample HR policy documents (PDF)
- Vector store for document retrieval
- Gradio-based chat interface
- Conversation memory

---
## Cleanup

In [ ]:
---
## Summary

In this notebook, we covered:

| Section | Key Concepts |
|---------|-------------|
| **Setup** | OpenRouter/OpenAI config, ChatOpenAI |
| **Fundamentals** | invoke, batch, stream, messages |
| **Prompts** | PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate |
| **Parsers** | StrOutputParser, JsonOutputParser, PydanticOutputParser |
| **Memory** | Buffer, Window, Summary, Entity memory types |
| **Chains** | LCEL, RunnablePassthrough, RunnableParallel, Sequential chains |
| **Document Loading** | PDF, CSV, TXT, TSV loaders |
| **Text Splitting** | RecursiveCharacterTextSplitter, chunk strategies |
| **Embeddings** | HuggingFaceEmbeddings, similarity comparison |
| **Vector Stores** | ChromaDB, similarity search, filtering |
| **Basic RAG** | Retrieval chain, multi-file support |
| **Advanced RAG** | MMR, compression, multi-query, conversation RAG |
| **Tools** | @tool decorator, StructuredTool, RAG as tool |
| **Agents** | create_openai_tools_agent, AgentExecutor, memory integration |
| **HR Chatbot** | PDF loading, Gradio UI, conversational RAG |

### Next Steps
- Explore LangGraph for more complex agent workflows
- Add more specialized tools (web search, SQL, APIs)
- Implement evaluation and monitoring
- Deploy as an API service

---
*Created for LangChain Workshop*

---
## Summary

In this notebook, we covered:

| Section | Key Concepts |
|---------|-------------|
| **Setup** | OpenRouter config, ChatOpenAI |
| **Fundamentals** | invoke, batch, stream, messages |
| **Prompts** | PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate |
| **Parsers** | StrOutputParser, JsonOutputParser, PydanticOutputParser |
| **Memory** | Buffer, Window, Summary, Entity memory types |
| **Chains** | LCEL, RunnablePassthrough, RunnableParallel, Sequential chains |
| **Document Loading** | PDF, CSV, TXT, TSV loaders |
| **Text Splitting** | RecursiveCharacterTextSplitter, chunk strategies |
| **Embeddings** | HuggingFaceEmbeddings, similarity comparison |
| **Vector Stores** | ChromaDB, similarity search, filtering |
| **Basic RAG** | Retrieval chain, multi-file support |
| **Advanced RAG** | MMR, compression, multi-query, conversation RAG |
| **Tools** | @tool decorator, StructuredTool, RAG as tool |
| **Agents** | create_openai_tools_agent, AgentExecutor, memory integration |

### Next Steps
- Explore LangGraph for more complex agent workflows
- Add more specialized tools (web search, SQL, APIs)
- Implement evaluation and monitoring
- Deploy as an API service

---
*Created for LangChain Workshop*

### 11.1 Create Sample HR Documents (PDF)

In [ ]:
from fpdf import FPDF
import os

# Create HR documents directory
os.makedirs("hr_documents", exist_ok=True)

# HR Document 1: Employee Handbook
employee_handbook = """
EMPLOYEE HANDBOOK
TechCorp Inc.
Last Updated: January 2024

CHAPTER 1: WELCOME TO TECHCORP

Welcome to TechCorp Inc.! We are excited to have you as part of our team. This handbook outlines 
our company policies, procedures, and benefits to help you succeed in your role.

CHAPTER 2: WORKING HOURS AND ATTENDANCE

Standard Working Hours:
- Regular working hours are Monday through Friday, 9:00 AM to 6:00 PM
- Core hours (mandatory presence) are 10:00 AM to 4:00 PM
- Flexible start time: Between 8:00 AM and 10:00 AM
- Flexible end time: Between 5:00 PM and 7:00 PM

Remote Work Policy:
- Employees may work remotely up to 3 days per week after completing 90-day probation
- Remote work requests must be submitted 24 hours in advance
- All remote workers must be available during core hours
- High-speed internet connection is required for remote work

Attendance:
- Employees must log attendance through the HR portal daily
- Tardiness of more than 15 minutes requires manager notification
- Three unexcused absences in a month may result in disciplinary action

CHAPTER 3: LEAVE POLICIES

Annual Leave:
- All full-time employees receive 20 days of paid annual leave per year
- Leave accrues at 1.67 days per month
- Maximum carry-forward: 5 days to the next year
- Leave requests must be submitted at least 2 weeks in advance for periods over 3 days

Sick Leave:
- Employees receive 12 days of paid sick leave per year
- Medical certificate required for absences exceeding 2 consecutive days
- Unused sick leave does not carry forward

Parental Leave:
- Maternity Leave: 16 weeks fully paid
- Paternity Leave: 4 weeks fully paid
- Adoption Leave: 12 weeks fully paid
- Additional unpaid leave up to 6 months available upon request

Other Leave Types:
- Bereavement Leave: 5 days for immediate family, 3 days for extended family
- Marriage Leave: 5 days
- Jury Duty: Fully paid for duration of service
- Voting Leave: 2 hours on election day

CHAPTER 4: COMPENSATION AND BENEFITS

Salary:
- Salaries are paid on the last working day of each month
- Direct deposit is mandatory for all employees
- Annual salary reviews occur in March
- Performance bonuses are paid in December

Health Insurance:
- Comprehensive health insurance coverage for employee and dependents
- Coverage includes medical, dental, and vision
- Company pays 80% of premium; employee pays 20%
- Coverage begins on the first day of employment

Retirement Benefits:
- 401(k) plan with company match up to 6% of salary
- Vesting schedule: 25% per year over 4 years
- Employees can contribute up to IRS maximum limits

Other Benefits:
- Life insurance: 2x annual salary
- Disability insurance: Short-term and long-term coverage
- Employee Assistance Program (EAP): Free counseling services
- Gym membership reimbursement: Up to $50/month
- Professional development budget: $2,000/year
- Stock options: Available for employees at manager level and above
"""

# HR Document 2: Code of Conduct
code_of_conduct = """
CODE OF CONDUCT
TechCorp Inc.

INTRODUCTION

This Code of Conduct outlines the principles and standards that guide our behavior at TechCorp. 
All employees are expected to read, understand, and comply with this code.

PROFESSIONAL BEHAVIOR

Respect and Dignity:
- Treat all colleagues, clients, and partners with respect and dignity
- Discrimination based on race, gender, age, religion, disability, or sexual orientation is prohibited
- Harassment of any kind will not be tolerated
- Report any violations to HR or use the anonymous ethics hotline

Workplace Safety:
- Follow all safety guidelines and procedures
- Report unsafe conditions immediately
- No weapons are permitted on company premises
- Drug and alcohol use during work hours is prohibited

CONFLICTS OF INTEREST

Employees must:
- Disclose any potential conflicts of interest to their manager and HR
- Not accept gifts valued over $100 from vendors or clients
- Not engage in outside employment that conflicts with TechCorp duties
- Not use company resources for personal gain

CONFIDENTIALITY

Protection of Information:
- Confidential company information must not be disclosed to unauthorized parties
- All employees must sign an NDA upon joining
- Customer data must be handled according to our privacy policy
- Breach of confidentiality may result in immediate termination

COMMUNICATION GUIDELINES

Internal Communication:
- Use professional language in all communications
- Respond to emails within 24 business hours
- Use appropriate channels: Slack for quick queries, email for formal communication
- Meeting invites should include agenda and relevant documents

External Communication:
- Only authorized spokespersons may speak to media
- Social media posts should not disclose confidential information
- Company logo use requires Marketing approval

DISCIPLINARY PROCESS

Violations of this code may result in:
1. Verbal Warning
2. Written Warning
3. Performance Improvement Plan
4. Suspension
5. Termination

The severity of action depends on the nature and frequency of violations.

REPORTING VIOLATIONS

- Report to immediate supervisor or HR
- Anonymous ethics hotline: 1-800-ETHICS-1
- Email: ethics@techcorp.com
- All reports are investigated confidentially
"""

# HR Document 3: Performance Review Policy
performance_policy = """
PERFORMANCE MANAGEMENT POLICY
TechCorp Inc.

OVERVIEW

TechCorp is committed to fostering employee growth through regular feedback and fair performance 
evaluations. This policy outlines our performance management process.

PERFORMANCE REVIEW CYCLE

Annual Reviews:
- Conducted in March each year
- Covers performance from January to December of previous year
- Self-assessment required before manager review
- Calibration meetings ensure fairness across departments

Mid-Year Check-ins:
- Conducted in August/September
- Focus on progress toward annual goals
- Opportunity to adjust goals if needed
- Not tied to compensation decisions

RATING SCALE

1 - Needs Improvement: Performance consistently below expectations
2 - Partially Meets: Performance occasionally meets expectations
3 - Meets Expectations: Consistently meets job requirements
4 - Exceeds Expectations: Frequently surpasses job requirements
5 - Outstanding: Exceptional performance, role model for others

GOAL SETTING

SMART Goals:
- Specific: Clear and well-defined
- Measurable: Quantifiable success criteria
- Achievable: Realistic given resources and constraints
- Relevant: Aligned with team and company objectives
- Time-bound: Clear deadline

Goal Categories:
- Business Goals: Revenue, efficiency, customer satisfaction
- Development Goals: Skills, certifications, leadership
- Collaboration Goals: Cross-team projects, mentoring

COMPENSATION DECISIONS

Merit Increases:
- Based on performance rating and market data
- Rating 1-2: 0% increase
- Rating 3: 2-4% increase
- Rating 4: 4-6% increase
- Rating 5: 6-10% increase

Promotions:
- Require consistent "Exceeds" or "Outstanding" ratings
- Manager recommendation and skip-level approval required
- Must demonstrate readiness for next-level responsibilities

PERFORMANCE IMPROVEMENT PLAN (PIP)

When Required:
- Rating of "Needs Improvement" for two consecutive periods
- Serious performance issues identified
- Manager and HR collaboration required

PIP Components:
- Clear statement of performance gaps
- Specific improvement actions and milestones
- 60-90 day duration
- Weekly check-ins with manager
- Resources and support provided

Outcomes:
- Successful completion: Return to normal performance management
- Unsuccessful: May result in termination
"""

# HR Document 4: IT and Security Policy
it_security_policy = """
IT AND SECURITY POLICY
TechCorp Inc.

PURPOSE

This policy establishes guidelines for the secure and appropriate use of TechCorp's information 
technology resources.

ACCEPTABLE USE

Permitted Uses:
- Business-related activities
- Limited personal use during breaks (non-disruptive)
- Professional development and learning

Prohibited Uses:
- Accessing inappropriate or illegal content
- Installing unauthorized software
- Sharing login credentials
- Using company email for personal business
- Cryptocurrency mining
- Torrenting or pirating content

PASSWORDS AND AUTHENTICATION

Password Requirements:
- Minimum 12 characters
- Must include uppercase, lowercase, numbers, and symbols
- Change passwords every 90 days
- No password reuse for last 10 passwords
- Use password manager (LastPass provided free)

Multi-Factor Authentication:
- Required for all company systems
- Use authenticator app (not SMS)
- Report lost authentication devices immediately

DATA SECURITY

Classification Levels:
- Public: Can be shared externally
- Internal: For employees only
- Confidential: Need-to-know basis
- Restricted: Highest sensitivity, special access required

Data Handling:
- Never store confidential data on personal devices
- Use encrypted drives for sensitive information
- Shred physical documents containing sensitive data
- Report data breaches within 24 hours

DEVICE MANAGEMENT

Company Devices:
- Laptops and phones are company property
- Return all devices upon termination
- Keep devices locked when unattended
- Enable full-disk encryption

BYOD (Bring Your Own Device):
- Must register with IT before accessing company resources
- MDM software installation required
- Company can remotely wipe company data
- Keep personal and work data separate

REMOTE ACCESS

VPN Requirements:
- Use company VPN for all remote work
- Never access company systems from public WiFi without VPN
- Report suspicious VPN activities

Home Office Security:
- Secure home WiFi with strong password
- Use separate network for work if possible
- Lock screen when stepping away
- Keep work discussions private (no eavesdroppers)

INCIDENT REPORTING

Report Immediately:
- Suspected phishing emails
- Lost or stolen devices
- Malware or virus detection
- Unauthorized access attempts
- Social engineering attempts

Contact IT Security:
- Email: security@techcorp.com
- Phone: Extension 5555
- Emergency: 24/7 security hotline
"""

def create_pdf(content: str, filename: str, title: str):
    """Create a PDF from text content."""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Title
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, title, ln=True, align="C")
    pdf.ln(10)
    
    # Content
    pdf.set_font("Helvetica", "", 10)
    
    for line in content.strip().split('\n'):
        # Check if it's a header (all caps or starts with CHAPTER)
        if line.strip().isupper() or line.strip().startswith('CHAPTER'):
            pdf.set_font("Helvetica", "B", 12)
            pdf.ln(5)
            pdf.multi_cell(0, 6, line.strip())
            pdf.set_font("Helvetica", "", 10)
        elif line.strip().startswith('-'):
            pdf.multi_cell(0, 5, "  " + line.strip())
        elif line.strip():
            pdf.multi_cell(0, 5, line.strip())
        else:
            pdf.ln(3)
    
    pdf.output(filename)
    print(f"Created: {filename}")

# Create all HR PDFs
create_pdf(employee_handbook, "hr_documents/employee_handbook.pdf", "Employee Handbook")
create_pdf(code_of_conduct, "hr_documents/code_of_conduct.pdf", "Code of Conduct")
create_pdf(performance_policy, "hr_documents/performance_policy.pdf", "Performance Management Policy")
create_pdf(it_security_policy, "hr_documents/it_security_policy.pdf", "IT and Security Policy")

print("\nAll HR documents created successfully!")

### 11.2 Build the HR RAG Chatbot

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
import os

class HRChatbot:
    """
    HR RAG Chatbot with conversation memory and document retrieval.
    """
    
    def __init__(self, llm, embeddings):
        self.llm = llm
        self.embeddings = embeddings
        self.vectorstore = None
        self.retriever = None
        self.rag_chain = None
        self.chat_history = []
        
    def load_hr_documents(self, documents_path: str = "hr_documents"):
        """Load all PDF documents from the HR documents folder."""
        print(f"Loading documents from {documents_path}...")
        
        # Load all PDFs from directory
        loader = DirectoryLoader(
            documents_path,
            glob="**/*.pdf",
            loader_cls=PyPDFLoader,
            show_progress=True
        )
        documents = loader.load()
        print(f"Loaded {len(documents)} pages from PDF documents")
        
        # Split documents into chunks
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        splits = text_splitter.split_documents(documents)
        print(f"Split into {len(splits)} chunks")
        
        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=splits,
            embedding=self.embeddings,
            persist_directory="./hr_vectorstore",
            collection_name="hr_policies"
        )
        
        # Create retriever with MMR for diversity
        self.retriever = self.vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": 5, "fetch_k": 10}
        )
        
        print("HR Knowledge base created successfully!")
        return self
    
    def build_chain(self):
        """Build the RAG chain with conversation history support."""
        
        # System prompt for HR chatbot
        system_prompt = """You are a helpful HR Assistant for TechCorp Inc. Your role is to answer 
employee questions about company policies, benefits, leave, performance reviews, and workplace guidelines.

Instructions:
- Answer questions based ONLY on the provided context from HR documents
- Be friendly, professional, and helpful
- If information is not in the context, say "I don't have information about that in our HR policies. Please contact HR directly at hr@techcorp.com"
- Provide specific details like numbers, dates, and procedures when available
- If the question is unclear, ask for clarification
- Format your responses clearly with bullet points when listing multiple items

Context from HR Documents:
{context}"""

        # Create the prompt with chat history
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}")
        ])
        
        def format_docs(docs):
            return "\n\n---\n\n".join(
                f"[Source: {doc.metadata.get('source', 'Unknown')}]\n{doc.page_content}" 
                for doc in docs
            )
        
        # Build the chain
        self.rag_chain = (
            {
                "context": self.retriever | format_docs,
                "chat_history": lambda x: x["chat_history"],
                "question": lambda x: x["question"]
            }
            | prompt
            | self.llm
            | StrOutputParser()
        )
        
        print("HR Chatbot chain built successfully!")
        return self
    
    def chat(self, message: str) -> str:
        """Process a message and return response."""
        if not self.rag_chain:
            return "Error: Chatbot not initialized. Please run build_chain() first."
        
        # Get response
        response = self.rag_chain.invoke({
            "question": message,
            "chat_history": self.chat_history
        })
        
        # Update chat history
        self.chat_history.append(HumanMessage(content=message))
        self.chat_history.append(AIMessage(content=response))
        
        # Keep only last 10 exchanges to prevent context overflow
        if len(self.chat_history) > 20:
            self.chat_history = self.chat_history[-20:]
        
        return response
    
    def clear_history(self):
        """Clear conversation history."""
        self.chat_history = []
        return "Conversation history cleared!"
    
    def get_sources(self, query: str) -> list:
        """Get source documents for a query."""
        if not self.retriever:
            return []
        docs = self.retriever.invoke(query)
        return [{"source": doc.metadata.get("source", "Unknown"), 
                 "content": doc.page_content[:200] + "..."} for doc in docs]


# Initialize the HR Chatbot
hr_chatbot = HRChatbot(llm, embeddings)
hr_chatbot.load_hr_documents("hr_documents")
hr_chatbot.build_chain()

In [ ]:
# Test the HR Chatbot
print("=== Testing HR Chatbot ===\n")

test_questions = [
    "How many days of annual leave do I get?",
    "What is the remote work policy?",
    "How does the performance review process work?",
    "What are the password requirements?",
    "How much maternity leave is provided?",
]

for question in test_questions:
    print(f"Q: {question}")
    response = hr_chatbot.chat(question)
    print(f"A: {response}\n")
    print("-" * 50)

### 11.3 Interactive Gradio Chat Interface

Run the cell below to launch an interactive chat UI for the HR Chatbot!

In [ ]:
import gradio as gr

# Reset chat history for fresh start
hr_chatbot.clear_history()

def respond(message, history):
    """Process user message and return response."""
    response = hr_chatbot.chat(message)
    return response

def clear_chat():
    """Clear the chat history."""
    hr_chatbot.clear_history()
    return [], ""

# Create Gradio interface with custom theme
with gr.Blocks(
    title="HR Assistant Chatbot",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="gray",
    ),
    css="""
    .gradio-container {
        max-width: 900px !important;
        margin: auto !important;
    }
    .chat-header {
        text-align: center;
        padding: 20px;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        border-radius: 10px;
        margin-bottom: 20px;
    }
    .chat-header h1 {
        color: white;
        margin: 0;
    }
    .chat-header p {
        color: #e0e0e0;
        margin: 10px 0 0 0;
    }
    """
) as demo:
    
    # Header
    gr.HTML("""
    <div class="chat-header">
        <h1>TechCorp HR Assistant</h1>
        <p>Ask me anything about company policies, benefits, leave, and more!</p>
    </div>
    """)
    
    # Chatbot interface
    chatbot = gr.Chatbot(
        height=450,
        show_label=False,
        avatar_images=(None, "https://api.dicebear.com/7.x/bottts/svg?seed=hr-bot"),
        bubble_full_width=False,
    )
    
    # Input area
    with gr.Row():
        msg = gr.Textbox(
            placeholder="Type your HR question here... (e.g., 'How many vacation days do I get?')",
            show_label=False,
            container=False,
            scale=9,
        )
        submit_btn = gr.Button("Send", variant="primary", scale=1)
    
    # Action buttons
    with gr.Row():
        clear_btn = gr.Button("Clear Chat", variant="secondary")
    
    # Example questions
    gr.Examples(
        examples=[
            "How many days of annual leave do I get?",
            "What is the remote work policy?",
            "How does performance review work?",
            "What are the health insurance benefits?",
            "What is the maternity leave policy?",
            "How do I report a security incident?",
            "What is the dress code?",
            "How do I request time off?",
        ],
        inputs=msg,
        label="Example Questions"
    )
    
    # Information accordion
    with gr.Accordion("About this HR Chatbot", open=False):
        gr.Markdown("""
        ### What can I help you with?
        
        This HR Assistant can answer questions about:
        - **Leave Policies**: Annual leave, sick leave, parental leave, etc.
        - **Benefits**: Health insurance, retirement, gym reimbursement
        - **Work Policies**: Remote work, working hours, attendance
        - **Performance**: Reviews, ratings, promotions, PIPs
        - **Code of Conduct**: Professional behavior, conflicts of interest
        - **IT Security**: Passwords, data handling, device policies
        
        ### Data Sources
        The chatbot uses the following HR documents:
        - Employee Handbook
        - Code of Conduct
        - Performance Management Policy
        - IT and Security Policy
        
        ### Note
        For sensitive or personal HR matters, please contact HR directly at hr@techcorp.com
        """)
    
    # Event handlers
    def user_submit(user_message, history):
        if not user_message.strip():
            return history, ""
        return history + [[user_message, None]], ""
    
    def bot_response(history):
        if not history or history[-1][1] is not None:
            return history
        user_message = history[-1][0]
        response = hr_chatbot.chat(user_message)
        history[-1][1] = response
        return history
    
    # Wire up events
    msg.submit(user_submit, [msg, chatbot], [chatbot, msg]).then(
        bot_response, chatbot, chatbot
    )
    submit_btn.click(user_submit, [msg, chatbot], [chatbot, msg]).then(
        bot_response, chatbot, chatbot
    )
    clear_btn.click(lambda: (hr_chatbot.clear_history(), [], "")[1:], outputs=[chatbot, msg])

# Launch the interface
demo.launch(share=False, debug=False)

### 11.4 Alternative: Simple Chat Interface

If you prefer a simpler interface, use this minimal version:

In [ ]:
# Simple Chat Interface (Alternative)
# Uncomment and run this cell if you want a simpler UI

# import gradio as gr

# # Reset history
# hr_chatbot.clear_history()

# def simple_chat(message, history):
#     """Simple chat function for Gradio ChatInterface."""
#     response = hr_chatbot.chat(message)
#     return response

# # Create simple chat interface
# simple_demo = gr.ChatInterface(
#     fn=simple_chat,
#     title="HR Assistant",
#     description="Ask questions about company policies, benefits, and procedures.",
#     examples=[
#         "How many vacation days do I get?",
#         "What is the remote work policy?",
#         "How do performance reviews work?",
#     ],
#     theme="soft",
# )

# simple_demo.launch()